In [ ]:
import os
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from webdriver_manager.chrome import ChromeDriverManager

from openpyxl import load_workbook
from openpyxl.styles import Alignment
from openpyxl.utils import get_column_letter


# =========================
# Excel保存路径
# =========================

desktop = os.path.join(
    os.path.expanduser("~"),
    "Desktop"
)

output_file = os.path.join(
    desktop,
    "小红书笔记数据.xlsx"
)


# =========================
# Chrome配置
# 与B站、抖音代码保持一致
# =========================

options = Options()

# 保存登录状态
options.add_argument(
    r"--user-data-dir=C:\Users\12082\AppData\Local\Google\Chrome\SeleniumData"
)

options.add_argument("--start-maximized")


# =========================
# 创建Chrome WebDriver
# =========================

try:

    driver = webdriver.Chrome(
        service=Service(
            ChromeDriverManager().install()
        ),
        options=options
    )

    print("Chrome启动成功")


except Exception as e:

    print(f"Chrome启动失败：{e}")

    exit()


# =========================
# 打开小红书创作中心
# =========================

url = "https://creator.xiaohongshu.com/new/note-manager"

try:

    driver.get(url)

    print("成功进入小红书创作中心")


except Exception as e:

    print(f"打开页面失败：{e}")

    driver.quit()

    exit()


# =========================
# 等待登录
# =========================

wait = WebDriverWait(driver, 300)

try:

    print("请扫码登录小红书创作中心...")

    wait.until(
        EC.presence_of_element_located(
            (
                By.CSS_SELECTOR,
                "div.note-card"
            )
        )
    )

    print("登录成功，进入笔记管理页面")


except Exception as e:

    print(f"登录失败或等待超时：{e}")

    driver.quit()

    exit()


# =========================
# 数字转换函数
# =========================

def convert_number(text):

    if not text:
        return 0

    text = text.strip()

    # 处理短横线等无数据情况
    if text in ["-", "--"]:
        return 0

    # 处理“万”
    if "万" in text:

        try:

            return int(
                float(
                    text.replace("万", "")
                ) * 10000
            )

        except ValueError:

            return 0

    # 处理逗号
    try:

        return int(
            text.replace(",", "")
        )

    except ValueError:

        return 0


# =========================
# 保存全部笔记
# =========================

all_note_data = []

# 用标题和日期共同去重
loaded_notes = set()

last_total = 0

no_change_count = 0


# =========================
# 自动滚动并提取全部笔记
# =========================

while True:

    # 查找当前已经加载的全部笔记卡片
    note_cards = driver.find_elements(
        By.CSS_SELECTOR,
        "div.note-card"
    )

    print(
        f"\n当前页面发现笔记：{len(note_cards)}"
    )


    for card in note_cards:

        try:

            # =========================
            # 提取标题
            # =========================

            title = card.find_element(
                By.CSS_SELECTOR,
                "span.note-card__title"
            ).text.strip()


            # =========================
            # 提取日期
            # =========================

            date = card.find_element(
                By.CSS_SELECTOR,
                "span.note-card__time"
            ).text.strip()


            if not title:

                continue


            # 标题和日期组合，防止同名笔记被误删
            unique_key = (
                title,
                date
            )


            if unique_key in loaded_notes:

                continue


            # =========================
            # 提取五项数据
            # =========================

            stat_elements = card.find_elements(
                By.CSS_SELECTOR,
                "div.note-card__stat span"
            )


            stat_values = [
                element.text.strip()
                for element in stat_elements
            ]


            # 默认全部为0
            view_count = 0
            comment_count = 0
            like_count = 0
            favorite_count = 0
            share_count = 0


            # 根据截图中图标顺序提取
            if len(stat_values) >= 1:

                view_count = convert_number(
                    stat_values[0]
                )


            if len(stat_values) >= 2:

                comment_count = convert_number(
                    stat_values[1]
                )


            if len(stat_values) >= 3:

                like_count = convert_number(
                    stat_values[2]
                )


            if len(stat_values) >= 4:

                favorite_count = convert_number(
                    stat_values[3]
                )


            if len(stat_values) >= 5:

                share_count = convert_number(
                    stat_values[4]
                )


            loaded_notes.add(
                unique_key
            )


            all_note_data.append(
                {
                    "笔记名称": title,
                    "发布日期": date,
                    "观看量": view_count,
                    "评论量": comment_count,
                    "点赞量": like_count,
                    "收藏量": favorite_count,
                    "分享量": share_count
                }
            )


            print(
                f"{title} | "
                f"日期：{date} | "
                f"观看：{view_count} | "
                f"评论：{comment_count} | "
                f"点赞：{like_count} | "
                f"收藏：{favorite_count} | "
                f"分享：{share_count}"
            )


        except Exception as e:

            print(
                f"单篇笔记提取失败：{e}"
            )


    # =========================
    # 判断是否已加载完
    # =========================

    current_total = len(note_cards)


    if current_total == last_total:

        no_change_count += 1

    else:

        no_change_count = 0


    last_total = current_total


    # 连续3次滚动后数量没有变化
    if no_change_count >= 3:

        print("\n已经加载全部笔记")

        break


    # =========================
    # 滚动到页面底部
    # =========================

    driver.execute_script(
        "window.scrollTo(0, document.body.scrollHeight);"
    )

    print("正在向下滚动，等待加载更多笔记...")

    time.sleep(3)


# =========================
# 关闭浏览器
# =========================

driver.quit()


# =========================
# 保存为DataFrame
# =========================

print(
    f"\n最终提取笔记数量：{len(all_note_data)}"
)


if not all_note_data:

    print("没有提取到任何笔记数据")

    exit()


df = pd.DataFrame(
    all_note_data
)


# 将数字列强制转换为整数
number_columns = [
    "观看量",
    "评论量",
    "点赞量",
    "收藏量",
    "分享量"
]

for column in number_columns:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    ).fillna(0).astype(int)


# =========================
# 保存Excel
# =========================

df.to_excel(
    output_file,
    index=False,
    sheet_name="笔记数据"
)


# =========================
# 调整Excel格式
# =========================

wb = load_workbook(
    output_file
)

ws = wb.active


# 设置列宽
ws.column_dimensions["A"].width = 45
ws.column_dimensions["B"].width = 22

for column_number in range(
    3,
    ws.max_column + 1
):

    column_letter = get_column_letter(
        column_number
    )

    ws.column_dimensions[
        column_letter
    ].width = 12


# 标题行居中
for cell in ws[1]:

    cell.alignment = Alignment(
        horizontal="center",
        vertical="center"
    )


# 日期和数字右对齐
for row in ws.iter_rows(
    min_row=2,
    min_col=2,
    max_col=ws.max_column,
    max_row=ws.max_row
):

    for cell in row:

        cell.alignment = Alignment(
            horizontal="right",
            vertical="center"
        )


wb.save(
    output_file
)


print(
    f"小红书笔记数据已保存到：{output_file}"
)


# =========================
# 自动打开Excel
# =========================

try:

    os.startfile(
        output_file
    )

except Exception as e:

    print(
        f"无法自动打开Excel：{e}"
    )